<a href="https://colab.research.google.com/github/Samoon19/NLP_Sentiment_YT/blob/main/NLP_Sentiment_yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer



data = pd.read_csv("dataset.csv")
print(data.head())
# Get features (column names)
features = data.columns
print("Features (Columns):")
print(features)

# Get number of samples (rows)
num_samples = data.shape[0]
print("\nNumber of Samples (Rows):", num_samples)

# Get number of features
num_features = data.shape[1]
print("Number of Features (Columns):", num_features)

                    CommentID      VideoID  \
0  UgyRjrEdJIPrf68uND14AaABAg  mcY4M9gjtsI   
1  UgxXxEIySAwnMNw8D7N4AaABAg  2vuXcw9SZbA   
2  UgxB0jh2Ur41mcXr5IB4AaABAg  papg2tsoFzg   
3  UgwMOh95MfK0GuXLLrF4AaABAg  31KTdfRH6nY   
4  UgxJuUe5ysG8OSbABAl4AaABAg  -hV6aeyPHPA   

                                          VideoTitle              AuthorName  \
0        They killed my friend.#tales #movie #shorts         @OneWhoWandered   
1  Man Utd conceding first penalty at home in yea...           @chiefvon3068   
2                       Welcome to Javascript Course          @Abdulla-ip8qr   
3  Building web applications in Java with Spring ...        @finnianthehuman   
4  After a new engine her car dies on her way hom...  @ryoutubeplaylistb6137   

            AuthorChannelID  \
0  UC_-UEXaBL1dqqUPGkDll49A   
1  UCZ1LcZESjYqzaQRhjdZJFwg   
2  UCWBK35w5Swy1iF5xIbEyw3A   
3  UCwQ2Z03nOcMxWozBb_Cv66w   
4  UCTTcJ0tsAKQokmHB2qVb1qQ   

                                         CommentText Se

In [16]:

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess(text):
    if isinstance(text, float):  # Handle NaN
        text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove symbols
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

# Apply preprocessing
data['cleaned'] = data['CommentText'].apply(preprocess)

# Print BEFORE and AFTER for first 5 rows
print("=== BEFORE PREPROCESSING ===")
print(data['CommentText'].head())

print("\n=== AFTER PREPROCESSING ===")
print(data['cleaned'].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


=== BEFORE PREPROCESSING ===
0                      Anyone know what movie this is?
1    The fact they're holding each other back while...
2                          waiting next video will be?
3    Thanks for the great video.\n\nI don't underst...
4    Good person helping good people.\nThis is how ...
Name: CommentText, dtype: object

=== AFTER PREPROCESSING ===
0                                    anyone know movie
1                 fact holding back equally aggressive
2                                   waiting next video
3    thanks great video understand db continues acc...
4    good person helping good people america except...
Name: cleaned, dtype: object


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Drop rows where 'Sentiment' is NaN before splitting
data.dropna(subset=['Sentiment'], inplace=True)
data.reset_index(drop=True, inplace=True) # Added to reset index and ensure alignment

X = data['cleaned']
y = data['Sentiment'] # Changed 'label' to 'Sentiment'

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
bow = CountVectorizer(
    max_features=5000,     # limit vocab size
    min_df=5,              # ignore rare words
    max_df=0.8             # ignore overly common words
)
def preprocess(text):
    if isinstance(text, float):
        text = str(text)
    text = text.lower()
        # remove repeated characters (e.g., aaaaa → aa)
    text = re.sub(r'(.)\1+', r'\1\1', text)

    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [38]:
print("BoW Training Shape:", X_train_bow.shape)
print("BoW Testing Shape:", X_test_bow.shape)

# Show some feature names
print("\nSample BoW Features:")
print(bow.get_feature_names_out()[:10])

# Convert a small part to readable format
print("\nBoW Vector (first 5 rows):")
print(X_train_bow[:5].toarray())

BoW Training Shape: (780427, 5000)
BoW Testing Shape: (195107, 5000)

Sample BoW Features:
['aa' 'aaj' 'aap' 'aapne' 'aaya' 'ab' 'abandoned' 'abhi' 'ability' 'able']

BoW Vector (first 5 rows):
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,     # limit vocab size
    min_df=5,              # ignore rare junk
    max_df=0.8,            # ignore overly common words
    ngram_range=(1,2)   # adds bigrams like "very good"

)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [40]:
print("TF-IDF Training Shape:", X_train_tfidf.shape)
print("TF-IDF Testing Shape:", X_test_tfidf.shape)

# Show feature names
print("\nSample TF-IDF Features:")
print(tfidf.get_feature_names_out()[:10])

# Show first 5 rows
print("\nTF-IDF Vector (first 5 rows):")
print(X_train_tfidf[:5].toarray())

TF-IDF Training Shape: (780427, 5000)
TF-IDF Testing Shape: (195107, 5000)

Sample TF-IDF Features:
['aa' 'aap' 'aapne' 'ab' 'ability' 'able' 'abortion' 'absolute'
 'absolutely' 'absolutely love']

TF-IDF Vector (first 5 rows):
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [41]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

# Train on BoW
model.fit(X_train_bow, y_train)
pred_bow = model.predict(X_test_bow)

# Train on TF-IDF
model.fit(X_train_tfidf, y_train)
pred_tfidf = model.predict(X_test_tfidf)

In [42]:
from sklearn.metrics import accuracy_score, classification_report

print("BoW Accuracy:", accuracy_score(y_test, pred_bow))
print("TF-IDF Accuracy:", accuracy_score(y_test, pred_tfidf))

print(classification_report(y_test, pred_tfidf))

BoW Accuracy: 0.5925774062437534
TF-IDF Accuracy: 0.6005883950857734
              precision    recall  f1-score   support

    Negative       0.54      0.69      0.61     65567
     Neutral       0.57      0.49      0.53     64776
    Positive       0.72      0.62      0.66     64764

    accuracy                           0.60    195107
   macro avg       0.61      0.60      0.60    195107
weighted avg       0.61      0.60      0.60    195107



In [ ]:
import time

start = time.time()
bow.fit_transform(X_train)
print("BoW Time:", time.time() - start)

start = time.time()
tfidf.fit_transform(X_train)
print("TF-IDF Time:", time.time() - start)